![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







# Import in apple verion
### Check Critical Package Version
✅ JAX: 0.6.2
✅ MuJoCo: 3.3.6
✅ Brax: 0.13.0
✅ Flax: 0.10.7

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from learning.notebooks.apple_mujoco_setup import *
import jax
import jax.numpy as jnp
import numpy as np
import mediapy
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics
from flax import serialization
from mujoco_playground import registry
import wandb
import os
import matplotlib.pyplot as plt
import contextlib
import io




In [ ]:
user = "weissma6-zhaw-school-of-engineering"
project = "UR10_pick_ppo"
run_id = "DomainR_off_lr0.0008_KPKV40030_20260207_132131_1950"  # Replace with your run

# --- Create output folder ---
os.makedirs("evaluation/downloaded_policies", exist_ok=True)

# --- Download artifact ---
api = wandb.Api()
run = api.run(f"{user}/{project}/{run_id}")

# Find the policy artifact
artifacts = run.logged_artifacts()
policy_artifact = None
for artifact in artifacts:
    if artifact.type == "model":
        policy_artifact = artifact
        break

if policy_artifact is None:
    raise ValueError(f"No model artifact found for run {run_id}")

print(f"Found artifact: {policy_artifact.name}")

# Download to local folder
artifact_dir = policy_artifact.download(root="downloaded_policies")
print(f"Downloaded to: {artifact_dir}")

# Load the parameters
params_path = os.path.join(artifact_dir, "params.msgpack")
with open(params_path, "rb") as f:
    params_bytes = f.read()

print(f"✓ Loaded params ({len(params_bytes) / 1024:.1f} KB)")
# --- Extract training config from wandb run ---
train_config = dict(run.config)
nf_params = train_config.get("network_factory", {})
if nf_params is None:
    nf_params = {}
print(f"✓ Training config loaded from wandb run")
print(f"  network_factory: {nf_params}")
print(f"  env_name:        {train_config.get('env_name', 'UR10PickCube')}")

# --- Extract training config from wandb run ---
train_config = dict(run.config)
nf_params = train_config.get("network_factory", {}) or {}
print(f"✓ Training config loaded")
print(f"  network_factory: {nf_params}")

In [ ]:
# Check top-level structure
import msgpack

with open(params_path, "rb") as f:
    raw = msgpack.unpackb(f.read(), raw=False, strict_map_key=False)

print(f"Top-level type: {type(raw)}")
print(f"Number of elements: {len(raw)}")
print(f"Keys/indices: {list(raw.keys()) if isinstance(raw, dict) else range(len(raw))}")

# Show structure of each top-level element
for i, item in enumerate(raw) if isinstance(raw, (list, tuple)) else raw.items():
    if isinstance(raw, dict):
        i, item = i, raw[i]
    print(f"\n=== Element {i} ===")
    if isinstance(item, dict):
        print(f"  Keys: {list(item.keys())}")
    else:
        print(f"  Type: {type(item)}")

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import mediapy
import os
import functools
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics
from flax import serialization
from mujoco_playground import registry

# --- Create output folder ---
os.makedirs("evaluation/graphs", exist_ok=True)

# --- Config (from training run) ---
env_name = train_config.get("env_name", "UR10PickCube")
episode_length = int(train_config.get("episode_length", 150))
seed = 42
video_path = "evaluation/graphs/"
video_tag = "rollout_video.mp4"
camera_kwargs = {"camera": "side_130", "width": 800, "height": 600}

# --- Load environment ---
env = registry.load(env_name)
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

# --- Rebuild network with SAME architecture as training ---
obs_size = env.observation_size
action_size = env.action_size
print(f"Obs size: {obs_size}, Action size: {action_size}")

normalize = running_statistics.normalize

nf_kwargs = {}
if nf_params:
    for k, v in nf_params.items():
        if isinstance(v, list):
            nf_kwargs[k] = tuple(v)
        else:
            nf_kwargs[k] = v
    print(f"  Network kwargs: {nf_kwargs}")

ppo_network = ppo_networks.make_ppo_networks(
    observation_size=obs_size,
    action_size=action_size,
    preprocess_observations_fn=normalize,
    **nf_kwargs,
)

# --- Build template & deserialize ---
rng = jax.random.PRNGKey(0)

dummy_normalizer_params = running_statistics.init_state(
    jax.ShapeDtypeStruct((obs_size,), jnp.float32)
)
dummy_policy_params = ppo_network.policy_network.init(rng)
dummy_value_params = ppo_network.value_network.init(rng)

params_template = {
    "0": dummy_normalizer_params,
    "1": dummy_policy_params,
    "2": dummy_value_params,
}

params = serialization.from_bytes(params_template, params_bytes)
normalizer_params = params["0"]
policy_params = params["1"]
value_params = params["2"]
print("✓ Params restored")

# --- Build policy THE SAME WAY as training ---
make_policy = ppo_networks.make_inference_fn(ppo_network)
inference_params = (normalizer_params, policy_params)
policy = jax.jit(make_policy(inference_params, deterministic=True))

# --- Run rollout ---
rng = jax.random.PRNGKey(seed)
rng, reset_rng = jax.random.split(rng)
state = jit_reset(reset_rng)

rollout = [state]
total_reward = 0.0
step_rewards = []

for step in range(episode_length):
    rng, act_rng = jax.random.split(rng)

    # Exactly like training: policy(obs, rng) -> (action, extras)
    act_out = policy(state.obs, act_rng)
    if isinstance(act_out, tuple):
        action = act_out[0]
    else:
        action = act_out
    action = jnp.asarray(action)

    state = jit_step(state, action)
    rollout.append(state)

    reward = float(state.reward)
    step_rewards.append(reward)
    total_reward += reward

# Statistics
step_rewards = np.array(step_rewards)
print(f"✓ Rollout complete")
print(f"  Total reward:      {total_reward:.2f}")
print(f"  Mean step reward:  {step_rewards.mean():.4f}")
print(f"  Min step reward:   {step_rewards.min():.4f}")
print(f"  Max step reward:   {step_rewards.max():.4f}")

# --- Render video ---
frames = env.render(rollout, **camera_kwargs)
frames = np.asarray(frames).astype(np.uint8)

fps = int(1.0 / env.dt)
full_video_path = os.path.join(video_path, video_tag)
# mediapy.write_video(full_video_path, frames, fps=fps)

mediapy.show_video(frames, fps=fps)

## Get mean reward of some number of rollouts

In [ ]:
# --- Multiple Rollouts for Statistics ---
num_rollouts = 50
seed_base = 100  # Different from video rollout seed

# Collect total rewards
total_rewards = []

print(f"Running {num_rollouts} rollouts...")

for i in range(num_rollouts):
    rng = jax.random.PRNGKey(seed_base + i)
    rng, reset_rng = jax.random.split(rng)
    state = jit_reset(reset_rng)
    
    episode_reward = 0.0
    
    for step in range(episode_length):
        rng, act_rng = jax.random.split(rng)
        
        # Use the same policy function built via make_policy
        act_out = policy(state.obs, act_rng)
        action = act_out[0] if isinstance(act_out, tuple) else act_out
        
        state = jit_step(state, action)
        episode_reward += float(state.reward)
    
    total_rewards.append(episode_reward)
    
    if (i + 1) % 10 == 0:
        print(f"  Completed {i + 1}/{num_rollouts} rollouts")

total_rewards = np.array(total_rewards)

print(f"\n✓ All rollouts complete")
print(f"  Mean reward:   {total_rewards.mean():.2f}")
print(f"  Std reward:    {total_rewards.std():.2f}")
print(f"  Min reward:    {total_rewards.min():.2f}")
print(f"  Max reward:    {total_rewards.max():.2f}")

# --- Histogram ---
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(total_rewards, bins=15, color="#2E86AB", edgecolor="white", alpha=0.8)

# Mean line
ax.axvline(
    total_rewards.mean(), 
    color="red", 
    linestyle="--", 
    linewidth=2, 
    label=f"Mean: {total_rewards.mean():.1f}"
)

# Std lines
ax.axvline(
    total_rewards.mean() - total_rewards.std(), 
    color="gray", 
    linestyle=":", 
    linewidth=1.5
)
ax.axvline(
    total_rewards.mean() + total_rewards.std(), 
    color="gray", 
    linestyle=":", 
    linewidth=1.5,
    label=f"± Std: {total_rewards.std():.1f}"
)

ax.set_xlabel("Total Episode Reward", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(f"Reward Distribution (n={num_rollouts})", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
# plt.savefig("evaluation/graphs/reward_histogram.pdf", bbox_inches="tight", dpi=300)
# plt.savefig("evaluation/graphs/reward_histogram.png", bbox_inches="tight", dpi=300)
plt.show()

# print(f"\n✓ Saved to evaluation/graphs/reward_histogram.pdf")

# Comparison between rollouts.

In [ ]:
# ═══════════════════════════════════════════════════════
# Robustness Comparison: Default vs Mass Shift vs Friction Shift
# ═══════════════════════════════════════════════════════
import mujoco
import functools

# --- Policies to compare (row per policy) ---
policy_runs = {
    # "label": "wandb_run_id"
    "No DR":   "DomainR_off_forcerange10_20260206_173852_9143",
    "With DR": "DomainR_MFR_lr0.0010_KPKV400030_20260206_194814_3655",
}

# --- Test conditions (column per condition) ---
# scale=1.0 means nominal; >1 = heavier/rougher, <1 = lighter/slippier
test_conditions = {
    "Default":         {"mass_scale": 1.0, "friction_scale": 1.0},
    "Mass ×2.0":       {"mass_scale": 2.0, "friction_scale": 1.0},
    "Friction ×0.5":   {"mass_scale": 1.0, "friction_scale": 0.5},
}

num_rollouts = 50
episode_length = 150
seed_base = 200

In [ ]:
def get_box_info(env):
    """Find box body/geom IDs and nominal values (same logic as training)."""
    mj_model = None
    for attr in ("mj_model", "_mj_model", "model"):
        obj = getattr(env, attr, None)
        if obj is not None and hasattr(obj, "ngeom"):
            mj_model = obj
            break
    if mj_model is None:
        sys = getattr(env, "sys", None)
        if sys is not None:
            mj_model = getattr(sys, "mj_model", None)
    if mj_model is None:
        raise RuntimeError("Cannot find mj_model on env")

    box_body_id = -1
    box_geom_id = -1
    for name in ("box", "cube", "object", "target_object", "pick_object"):
        if box_body_id < 0:
            box_body_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, name)
        if box_geom_id < 0:
            box_geom_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, name)

    assert box_body_id >= 0, f"Box body not found"
    assert box_geom_id >= 0, f"Box geom not found"

    return {
        "box_body_id": box_body_id,
        "box_geom_id": box_geom_id,
        "nominal_mass": float(mj_model.body_mass[box_body_id]),
        "nominal_friction": mj_model.geom_friction[box_geom_id].copy(),
    }


def load_env_with_shift(env_name, box_info, mass_scale=1.0, friction_scale=1.0):
    """Load a fresh env and patch the raw MuJoCo model with shifted mass/friction."""
    env = registry.load(env_name)

    # Find the raw mujoco.MjModel (same logic as _get_mj_model in training)
    mj_model = None
    for attr in ("mj_model", "_mj_model", "model"):
        obj = getattr(env, attr, None)
        if obj is not None and hasattr(obj, "ngeom"):
            mj_model = obj
            break
    if mj_model is None:
        sys = getattr(env, "sys", None)
        if sys is not None:
            mj_model = getattr(sys, "mj_model", None)
    if mj_model is None:
        raise RuntimeError(f"Cannot find mj_model on env (type={type(env)})")

    # Patch in-place on the raw MuJoCo model
    bid = box_info["box_body_id"]
    gid = box_info["box_geom_id"]

    if mass_scale != 1.0:
        mj_model.body_mass[bid] = box_info["nominal_mass"] * mass_scale

    if friction_scale != 1.0:
        mj_model.geom_friction[gid][0] = box_info["nominal_friction"][0] * friction_scale

    return env


def load_policy_from_run(run_id, env):
    """Download policy from wandb and build the inference function."""
    api = wandb.Api()
    run = api.run(f"{user}/{project}/{run_id}")
    train_cfg = dict(run.config)
    nf_params = train_cfg.get("network_factory", {}) or {}

    # Download params
    artifacts = run.logged_artifacts()
    policy_art = next((a for a in artifacts if a.type == "model"), None)
    if policy_art is None:
        raise ValueError(f"No model artifact for {run_id}")
    art_dir = policy_art.download(root="downloaded_policies")

    with open(os.path.join(art_dir, "params.msgpack"), "rb") as f:
        params_bytes = f.read()

    # Rebuild network with correct architecture
    nf_kwargs = {}
    for k, v in nf_params.items():
        nf_kwargs[k] = tuple(v) if isinstance(v, list) else v

    obs_size = env.observation_size
    action_size = env.action_size

    ppo_net = ppo_networks.make_ppo_networks(
        observation_size=obs_size,
        action_size=action_size,
        preprocess_observations_fn=running_statistics.normalize,
        **nf_kwargs,
    )

    # Deserialize
    rng = jax.random.PRNGKey(0)
    template = {
        "0": running_statistics.init_state(jax.ShapeDtypeStruct((obs_size,), jnp.float32)),
        "1": ppo_net.policy_network.init(rng),
        "2": ppo_net.value_network.init(rng),
    }
    params = serialization.from_bytes(template, params_bytes)

    # Build inference fn (same as training)
    make_policy = ppo_networks.make_inference_fn(ppo_net)
    inference_params = (params["0"], params["1"])
    policy_fn = jax.jit(make_policy(inference_params, deterministic=True))

    return policy_fn


def run_rollouts(policy_fn, env_name, box_info, num_rollouts, episode_length, seed_base,
                 mass_range=None, friction_range=None, n_bins=5):
    """Run N rollouts. If ranges given, use discrete bins to avoid re-JIT per rollout."""
    rewards = []

    if mass_range is None and friction_range is None:
        # Default: single env, single JIT, all rollouts
        bins = [{"mass_scale": 1.0, "friction_scale": 1.0, "n": num_rollouts, "seed_offset": 0}]
    else:
        # Spread rollouts across n_bins discrete physics settings
        rolls_per_bin = num_rollouts // n_bins
        bins = [] 
        rng_np = np.random.RandomState(seed_base)
        for b in range(n_bins):
            ms = rng_np.uniform(*mass_range) if mass_range else 1.0
            fs = rng_np.uniform(*friction_range) if friction_range else 1.0
            bins.append({
                "mass_scale": ms, "friction_scale": fs,
                "n": rolls_per_bin, "seed_offset": b * rolls_per_bin,
            })

    for b in bins:
        with contextlib.redirect_stdout(io.StringIO()):
            test_env = load_env_with_shift(
                env_name, box_info,
                mass_scale=b["mass_scale"],
                friction_scale=b["friction_scale"],
            )
        # JIT once per bin
        jit_reset = jax.jit(test_env.reset)
        jit_step = jax.jit(test_env.step)

        for i in range(b["n"]):
            seed_i = seed_base + b["seed_offset"] + i
            rng = jax.random.PRNGKey(seed_i)
            rng, reset_rng = jax.random.split(rng)
            state = jit_reset(reset_rng)
            ep_reward = 0.0

            for _ in range(episode_length):
                rng, act_rng = jax.random.split(rng)
                act_out = policy_fn(state.obs, act_rng)
                action = act_out[0] if isinstance(act_out, tuple) else act_out
                state = jit_step(state, jnp.asarray(action))
                ep_reward += float(state.reward)

            rewards.append(ep_reward)

    return np.array(rewards)
print("✓ Helper functions defined")

In [33]:
test_conditions = {
    "Default":              {"mass_range": None,       "friction_range": None},
    "Mass [0.5×–2.0×]":    {"mass_range": (0.5, 2.0), "friction_range": None},
    "Friction [0.5×–2.0×]":{"mass_range": None,        "friction_range": (0.5, 2.0)},
}
# --- Get box info from a default env (once) ---
ref_env = registry.load(env_name)
box_info = get_box_info(ref_env)
print(f"Box body id: {box_info['box_body_id']}, "
      f"nominal mass: {box_info['nominal_mass']:.4f}, "
      f"nominal friction: {box_info['nominal_friction']}")

# --- Run the grid ---
# results[policy_label][condition_label] = np.array of rewards
results = {}

for policy_label, rid in policy_runs.items():
    print(f"\n{'='*60}")
    print(f"Policy: {policy_label}  ({rid})")
    print(f"{'='*60}")
    results[policy_label] = {}

    # Load policy once (architecture is the same across conditions)
    policy_fn = load_policy_from_run(rid, ref_env)

    for cond_label, cond in test_conditions.items():
        print(f"  Condition: {cond_label} ... ", end="", flush=True)
        rewards = run_rollouts(
            policy_fn, env_name, box_info, num_rollouts, episode_length, seed_base,
            mass_range=cond.get("mass_range"),
            friction_range=cond.get("friction_range"),
        )
        results[policy_label][cond_label] = rewards
        print(f"mean={rewards.mean():.1f} ± {rewards.std():.1f}")

print("\n✓ All comparisons complete")

KeyboardInterrupt: 

In [ ]:
policy_labels = list(results.keys())
cond_labels = list(test_conditions.keys())
n_rows = len(policy_labels)
n_cols = len(cond_labels)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows),
                         sharex=True, sharey=True, squeeze=False)

# Global min/max for consistent x-axis
all_rewards = [r for p in results.values() for r in p.values()]
global_min = min(r.min() for r in all_rewards)
global_max = max(r.max() for r in all_rewards)
pad = (global_max - global_min) * 0.1
bins = np.linspace(global_min - pad, global_max + pad, 20)

for row, policy_label in enumerate(policy_labels):
    for col, cond_label in enumerate(cond_labels):
        ax = axes[row][col]
        rews = results[policy_label][cond_label]

        ax.hist(rews, bins=bins, color="#2E86AB", edgecolor="white", alpha=0.8)
        ax.axvline(rews.mean(), color="red", linestyle="--", linewidth=2,
                   label=f"μ={rews.mean():.1f}")
        ax.axvline(rews.mean() - rews.std(), color="gray", linestyle=":", linewidth=1.2)
        ax.axvline(rews.mean() + rews.std(), color="gray", linestyle=":", linewidth=1.2,
                   label=f"σ={rews.std():.1f}")
        ax.legend(fontsize=8)

        if row == 0:
            ax.set_title(cond_label, fontsize=13, fontweight="bold")
        if col == 0:
            ax.set_ylabel(policy_label, fontsize=12, fontweight="bold")
        if row == n_rows - 1:
            ax.set_xlabel("Episode Reward", fontsize=10)

fig.suptitle(f"Robustness Comparison ({num_rollouts} rollouts each)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()